# App de Recomendación de Cocinas 🍳

En esta lección final de clasificación, vamos a:
1. Entrenar un modelo de clasificación
2. Guardarlo en formato ONNX (para producción)
3. Crear una app web que use el modelo en el navegador

Es el primer paso hacia **sistemas de recomendación** reales.

## 1. Instalar dependencias

Necesitamos `skl2onnx` para convertir nuestro modelo de Scikit-learn
al formato ONNX (Open Neural Network Exchange).

### ¿Qué es ONNX?

ONNX es un formato estándar para guardar modelos ML. Permite:
- Ejecutar modelos en **cualquier plataforma** (navegador, móvil, servidor)
- Usar diferentes frameworks (sklearn, PyTorch, TensorFlow)
- Optimizar modelos para producción

In [1]:
!pip install skl2onnx

  Using cached protobuf-7.35.1-cp310-abi3-win_amd64.whl (439 kB)


You should consider upgrading via the 'C:\Users\mikel\Documents\ML-Microsoft\ML-For-Beginners\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


## 2. Cargar y preparar datos

Usamos el mismo dataset limpio de las lecciones anteriores:
recetas de 5 cocinas asiáticas con 380 ingredientes.

In [2]:
import pandas as pd

data = pd.read_csv('../data/cleaned_cuisines.csv')
data.head()

,Unnamed: 0,cuisine,almond,angelica,anise,anise_seed,apple,apple_brandy,apricot,armagnac,...,whiskey,white_bread,white_wine,whole_grain_wheat_flour,wine,wood,yam,yeast,yogurt,zucchini
0,0,indian,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,indian,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2,indian,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,3,indian,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,4,indian,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


### Separar features y labels

- **X** (features): Los 380 ingredientes (columnas 2 en adelante)
- **y** (label): La cocina (columna `cuisine`)

Quitamos las dos primeras columnas (`Unnamed: 0` y `cuisine`) porque
no son ingredientes.

In [3]:
X = data.iloc[:, 2:]
X.head()

,almond,angelica,anise,anise_seed,apple,apple_brandy,apricot,armagnac,artemisia,artichoke,...,whiskey,white_bread,white_wine,whole_grain_wheat_flour,wine,wood,yam,yeast,yogurt,zucchini
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


In [4]:
y = data[['cuisine']]
y.head()

,cuisine
0,indian
1,indian
2,indian
3,indian
4,indian


## 3. Entrenar el modelo SVC

Usamos **Support Vector Classifier** con kernel lineal porque
en la lección anterior vimos que funcionaba bien (~79% accuracy).

### Parámetros clave
- `kernel='linear'`: Frontera de decisión lineal
- `C=10`: Regularización (controla la complejidad)
- `probability=True`: Necesario para convertir a ONNX

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

# Entrenar modelo
model = SVC(kernel='linear', C=10, probability=True, random_state=0)
model.fit(X_train, y_train.values.ravel())

,C,10
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,True
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


## 4. Evaluar el modelo

Antes de exportar, verificamos que el modelo funcione bien.

In [6]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

     chinese       0.69      0.65      0.67       243
      indian       0.89      0.89      0.89       214
    japanese       0.70      0.73      0.72       244
      korean       0.82      0.77      0.79       261
        thai       0.77      0.82      0.80       237

    accuracy                           0.77      1199
   macro avg       0.77      0.77      0.77      1199
weighted avg       0.77      0.77      0.77      1199



## 5. Convertir a ONNX

### ¿Por qué convertir?

El modelo de sklearn solo funciona en Python. Para usarlo en una
app web (JavaScript), necesitamos el formato ONNX.

### ¿Qué es un Tensor?

Un tensor es la estructura de datos que el modelo recibe.
En nuestro caso, es un vector de 380 números (0 o 1) que
representan los ingredientes.

`FloatTensorType([None, 380])` significa:
- `None`: Cualquier cantidad de ejemplos
- `380`: 380 features (ingredientes)

In [7]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# Definir el tipo de entrada (380 ingredientes)
initial_type = [('float_input', FloatTensorType([None, 380]))]

# Opciones de conversión
options = {id(model): {'nocl': True, 'zipmap': False}}

# Convertir
onx = convert_sklearn(model, initial_types=initial_type, options=options)

### Explicación de opciones

| Opción | Qué hace |
|--------|----------|
| `nocl=True` | Elimina información de clases del modelo (reduce tamaño) |
| `zipmap=False` | No genera diccionario de probabilidades (más rápido) |

In [8]:
# Guardar modelo
with open('./model.onnx', 'wb') as f:
    f.write(onx.SerializeToString())

print('model.onnx guardado exitosamente')

model.onnx guardado exitosamente


## 6. Visualizar el modelo (opcional)

Podés usar [Netron](https://github.com/lutzroeder/Netron)
para ver la estructura de tu modelo ONNX.

1. Descargá Netron
2. Abrí el archivo `model.onnx`
3. Vas a ver: 380 entradas → SVC → predicción

Es útil para verificar que el modelo esté bien construido.

---

## 7. Crear la app web

Ahora creamos un archivo `index.html` que:
1. Muestra checkboxes de ingredientes
2. Cuando el usuario marca ingredientes y hace clic, carga el modelo ONNX
3. Predice qué cocina puede hacer

### Arquitectura

```
Navegador → Carga model.onnx → Recibe ingredientes → Predice cocina
```

**Sin servidor Python** — todo corre en el navegador.

In [9]:
# Crear el archivo HTML
html_content = '''
<!DOCTYPE html>
<html>
    <header>
        <title>Cocinero IA</title>
    </header>
    <body>
        <h1>Revisa tu refrigerador. ¿Qué puedes crear?</h1>
        <div id="wrapper">
            <div class="boxCont">
                <input type="checkbox" value="4" class="checkbox">
                <label>Manzana</label>
            </div>
            <div class="boxCont">
                <input type="checkbox" value="247" class="checkbox">
                <label>Pera</label>
            </div>
            <div class="boxCont">
                <input type="checkbox" value="77" class="checkbox">
                <label>Cereza</label>
            </div>
            <div class="boxCont">
                <input type="checkbox" value="126" class="checkbox">
                <label>Fenogreco</label>
            </div>
            <div class="boxCont">
                <input type="checkbox" value="302" class="checkbox">
                <label>Sake</label>
            </div>
            <div class="boxCont">
                <input type="checkbox" value="327" class="checkbox">
                <label>Salsa de soja</label>
            </div>
            <div class="boxCont">
                <input type="checkbox" value="112" class="checkbox">
                <label>Comino</label>
            </div>
        </div>
        <div style="padding-top:10px">
            <button onClick="startInference()">¿Qué tipo de cocina puedes preparar?</button>
        </div>

        <script src="https://cdn.jsdelivr.net/npm/onnxruntime-web@1.9.0/dist/ort.min.js"></script>
        <script>
            const ingredients = Array(380).fill(0);
            
            const checks = [...document.querySelectorAll('.checkbox')];
            
            checks.forEach(check => {
                check.addEventListener('change', function() {
                    ingredients[check.value] = check.checked ? 1 : 0;
                });
            });

            function testCheckboxes() {
                return checks.some(check => check.checked);
            }

            async function startInference() {
                let atLeastOneChecked = testCheckboxes()
                
                if (!atLeastOneChecked) {
                    alert('Por favor selecciona al menos un ingrediente.');
                    return;
                }
                try {
                    const session = await ort.InferenceSession.create('./model.onnx');
                    const input = new ort.Tensor(new Float32Array(ingredients), [1, 380]);
                    const feeds = { float_input: input };
                    const results = await session.run(feeds);
                    alert('¡Hoy puedes disfrutar de cocina ' + results.label.data[0] + '!')
                } catch (e) {
                    console.log('failed to inference ONNX model');
                    console.error(e);
                }
            }
        </script>
    </body>
</html>
'''

with open('./index.html', 'w') as f:
    f.write(html_content)

print('index.html creado exitosamente')

index.html creado exitosamente


### Explicación del código JavaScript

| Línea | Qué hace |
|-------|----------|
| `Array(380).fill(0)` | Crea vector de 380 ceros (ingredientes no marcados) |
| `querySelectorAll('.checkbox')` | Busca todos los checkboxes |
| `ingredients[check.value] = 1` | Marca el ingrediente como presente |
| `ort.InferenceSession.create()` | Carga el modelo ONNX |
| `new ort.Tensor(...)` | Crea el tensor de entrada |
| `session.run(feeds)` | Ejecuta la inferencia |
| `results.label.data[0]` | Obtiene la predicción |

### ¿Cómo funciona la inferencia?

1. El usuario marca checkboxes
2. Se crea un vector de 380 elementos (0 o 1)
3. Se carga el modelo ONNX en el navegador
4. Se envía el tensor al modelo
5. El modelo devuelve la cocina predicha

## 8. Ejecutar la app

Para probar la app:
1. Abrí una terminal en la carpeta del notebook
2. Ejecutá `npx http-server` o `python -m http.server 8000`
3. Abrí `http://localhost:8000` en tu navegador
4. Marcá ingredientes y hacé clic en el botón

---

## ✅ Resumen

| Paso | Qué hicimos | Herramienta |
|------|-------------|-------------|
| 1 | Cargar datos limpios | pandas |
| 2 | Entrenar modelo SVC | sklearn |
| 3 | Evaluar con classification report | sklearn |
| 4 | Convertir a ONNX | skl2onnx |
| 5 | Crear app web HTML/JS | Onnx Runtime |

### Conceptos clave

| Término | Significado |
|---------|-------------|
| **ONNX** | Formato estándar para modelos ML en producción |
| **Tensor** | Estructura de datos que el modelo recibe |
| **Inferencia** | Usar el modelo para predecir |
| **skl2onnx** | Librería para convertir sklearn a ONNX |
| **Onnx Runtime** | Librería para ejecutar modelos ONNX |

### Arquitectura final

```
Datos → sklearn → modelo → skl2onnx → model.onnx → navegador → predicción
```